In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

# Milestone 3: SABR Smile Module

This notebook studies how SABR generates strike-dependent implied Black volatilities and how those smile-adjusted volatilities change swaption prices relative to the flat-vol Black benchmark.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from swaption_pricing.black76 import price_swaption
from swaption_pricing.curve_bootstrap import bootstrap_zero_curve
from swaption_pricing.sabr import SabrParams, price_swaption_with_sabr, sabr_implied_volatility, sabr_vol_surface_slice
from swaption_pricing.swap import forward_swap_rate
from swaption_pricing.types import MarketQuote, SwaptionSpec

## 1. SABR Definitions

SABR adds a volatility model on top of the Black pricing layer.

- `alpha`: overall vol level
- `beta`: level dependence of the forward rate
- `rho`: skew control through correlation
- `nu`: vol-of-vol, which controls smile curvature

In this project, SABR does not replace Black pricing. Instead, SABR generates implied Black volatility for each strike, and the Black formula still performs the final pricing step.

## 2. Build the Curve and Base SABR Setup

In [ ]:
quotes = [
    MarketQuote(instrument_type='deposit', maturity=1.0, rate=0.0420),
    MarketQuote(instrument_type='swap', maturity=2.0, rate=0.0415, pay_frequency=1),
    MarketQuote(instrument_type='swap', maturity=3.0, rate=0.0410, pay_frequency=1),
    MarketQuote(instrument_type='swap', maturity=4.0, rate=0.0408, pay_frequency=1),
    MarketQuote(instrument_type='swap', maturity=5.0, rate=0.0405, pay_frequency=1),
    MarketQuote(instrument_type='swap', maturity=6.0, rate=0.0403, pay_frequency=1),
    MarketQuote(instrument_type='swap', maturity=7.0, rate=0.0402, pay_frequency=1),
]
curve = bootstrap_zero_curve(quotes)
base_params = SabrParams(alpha=0.0200, beta=0.50, rho=-0.25, nu=0.40)
flat_black_vol = 0.20
expiry = 2.0
tenor = 5.0
pay_frequency = 1
strikes = [0.0300, 0.0350, 0.0400, 0.0450, 0.0500]
forward = forward_swap_rate(curve, expiry, tenor, pay_frequency)
forward

## 3. SABR Smile Slice

Here we generate strike-by-strike SABR implied volatilities for one expiry-tenor point.

In [ ]:
smile_df = pd.DataFrame(
    sabr_vol_surface_slice(curve, expiry, tenor, pay_frequency, strikes, base_params),
    columns=['strike', 'sabr_vol'],
)
smile_df['flat_black_vol'] = flat_black_vol
smile_df

In [ ]:
ax = smile_df.plot(x='strike', y=['flat_black_vol', 'sabr_vol'], marker='o', figsize=(8, 4), title='Flat Black Vol vs SABR Smile')
ax.set_ylabel('Implied Volatility')
plt.show()

## 4. Black vs SABR Pricing Comparison

We now feed SABR-implied volatilities into the Black pricing formula and compare prices against the flat-vol benchmark.

In [ ]:
rows = []
for strike in strikes:
    spec = SwaptionSpec(
        notional=10_000_000.0,
        expiry=expiry,
        tenor=tenor,
        strike=strike,
        pay_frequency=pay_frequency,
        option_type='payer',
    )
    black_price = price_swaption(curve, spec, flat_black_vol)
    sabr_price, sabr_vol = price_swaption_with_sabr(curve, spec, base_params)
    rows.append({
        'strike': strike,
        'black_price': black_price,
        'sabr_price': sabr_price,
        'sabr_vol': sabr_vol,
        'price_diff': sabr_price - black_price,
    })

pricing_df = pd.DataFrame(rows)
pricing_df

In [ ]:
ax = pricing_df.plot(x='strike', y=['black_price', 'sabr_price'], marker='o', figsize=(8, 4), title='Black vs SABR Price by Strike')
ax.set_ylabel('Swaption Price')
plt.show()

ax = pricing_df.plot(x='strike', y='price_diff', kind='bar', figsize=(8, 4), title='SABR Price Minus Black Price')
ax.set_ylabel('Price Difference')
plt.xticks(rotation=0)
plt.show()

## 5. Receiver Comparison

The smile effect also matters for receiver swaptions. We compare Black and SABR prices using the same strike set.

In [ ]:
receiver_rows = []
for strike in strikes:
    spec = SwaptionSpec(
        notional=10_000_000.0,
        expiry=expiry,
        tenor=tenor,
        strike=strike,
        pay_frequency=pay_frequency,
        option_type='receiver',
    )
    black_price = price_swaption(curve, spec, flat_black_vol)
    sabr_price, sabr_vol = price_swaption_with_sabr(curve, spec, base_params)
    receiver_rows.append({
        'strike': strike,
        'black_price': black_price,
        'sabr_price': sabr_price,
        'sabr_vol': sabr_vol,
        'price_diff': sabr_price - black_price,
    })

receiver_df = pd.DataFrame(receiver_rows)
receiver_df

## 6. Parameter Sensitivity

We isolate the SABR parameters one at a time to see how they affect the smile shape.

In [ ]:
parameter_sets = {
    'base': base_params,
    'higher_alpha': SabrParams(alpha=0.0300, beta=0.50, rho=-0.25, nu=0.40),
    'less_negative_rho': SabrParams(alpha=0.0200, beta=0.50, rho=0.10, nu=0.40),
    'higher_nu': SabrParams(alpha=0.0200, beta=0.50, rho=-0.25, nu=0.70),
}

param_rows = []
for label, params in parameter_sets.items():
    for strike in strikes:
        param_rows.append({
            'scenario': label,
            'strike': strike,
            'sabr_vol': sabr_implied_volatility(forward, strike, expiry, params),
        })

param_df = pd.DataFrame(param_rows)
param_df.head()

In [ ]:
pivot = param_df.pivot(index='strike', columns='scenario', values='sabr_vol')
pivot.plot(marker='o', figsize=(9, 5), title='SABR Parameter Sensitivity by Strike')
plt.ylabel('Implied Volatility')
plt.show()

## 7. Interpretation

- SABR does not replace Black pricing; it supplies strike-dependent implied Black volatility.
- `alpha` mainly shifts the overall level of the smile.
- `rho` changes skew, so the two wings move asymmetrically.
- `nu` changes curvature, making the smile more or less pronounced.
- The impact on implied vol translates directly into different swaption prices across strikes.

This completes the first usable SABR smile module before moving into calibration-focused work.